<h2>Data Manipulation with Pandas</h2>

Pandas is a package built on top of NumPy, and provides an efficient implementation of a `DataFrame`. `DataFrames` are essentially multidimensional arrays with attached row and column labels, and often with heterogenous types and/or missing data. As well as offering a convenient storage interface for labeled data, Pandas implements a number of powerful data operations familiar to users of both database frameworks and spreadsheet programs.

NumPy's `ndarray` data structure provides essential features for the type of clean, well-organized data typically seen in numerical computing tasks. While it serves this purpose very well, its limitations become clear when we need more flexibility (attaching labels to data, working with missing data, etc.) and when attempting operations that do not map well to element-wise broadcasting (groupings, pivots, etc.), each of which is an important piece of analyzing the less structured data available in many forms in the world around us.

We'll explore the mechanics of using `Series`, `DataFrame`, and related structures effectively. These examples are from real datasets where appropriate.

In [1]:
import pandas

pandas.__version__

'1.2.4'

In [3]:
import pandas as pd

# Recall that you can easily access documentation with a question mark, or pd.<TAB>
pd?

Type:        module
String form: <module 'pandas' from '/Users/mike/opt/anaconda3/lib/python3.8/site-packages/pandas/__init__.py'>
File:        ~/opt/anaconda3/lib/python3.8/site-packages/pandas/__init__.py
Docstring:  
pandas - a powerful data analysis and manipulation library for Python

**pandas** is a Python package providing fast, flexible, and expressive data
structures designed to make working with "relational" or "labeled" data both
easy and intuitive. It aims to be the fundamental high-level building block for
doing practical, **real world** data analysis in Python. Additionally, it has
the broader goal of becoming **the most powerful and flexible open source data
analysis / manipulation tool available in any language**. It is already well on
its way toward this goal.

Main Features
-------------
Here are just a few of the things that pandas does well:

  - Easy handling of missing data in floating point as well as non-floating
    point data.
  - Size mutability: columns can 

In [5]:
import numpy as np

data = pd.Series([0.25, 0.5, 0.75, 1.0])
data

0    0.25
1    0.50
2    0.75
3    1.00
dtype: float64

The `Series` wraps both a sequence of values and a sequence of indices, which we can access with the `values` and `index` attributes. The `values` are simply a familiar NumPy array.

In [6]:
data.values

array([0.25, 0.5 , 0.75, 1.  ])

In [7]:
data.index

RangeIndex(start=0, stop=4, step=1)

In [8]:
type(data.values)

numpy.ndarray

Like with a NumPy array, data can be accessed by the associated index via the familiar Python square-bracket notation:

In [9]:
data[1]

0.5

In [10]:
data[1:3]

1    0.50
2    0.75
dtype: float64

The Pandas `Series` is much more general and flexible than the one-dimensional NumPy array that it emulates.

From what we've seen so far, it may look like the `Series` object is basically interchangeable with a one-dimensional NumPy array. The essential difference is the presence of the index: while the NumPy array has an <i>implicitly defined</i> integer index used to access the values, the Pandas `Series` has an <i>explicitly defined</i> index associated with the values.

The explicit index definition gives the `Series` object additional capabilities. For example, the index need not be an integer, but can consist of values of any desired type. Example of using strings as an index:

In [11]:
data = pd.Series([0.25, 0.5, 0.75, 1.0], index=['a', 'b', 'c', 'd'])
data

a    0.25
b    0.50
c    0.75
d    1.00
dtype: float64

In [12]:
data['b']

0.5

We can even use noncontiguous or nonsequential indices:

In [17]:
data = pd.Series([0.25, 0.5, 0.75, 1.0], index=[2, 5, 3, 7])
data

2    0.25
5    0.50
3    0.75
7    1.00
dtype: float64

In [18]:
data[5]

0.5

<b>Series as a specialized dictionary</b>

You can think of a Pandas `Series` a bit like a specialization of a Python dictionary. A dictionary is a structure that maps arbitrary keys to a set of arbitrary values, and a `Series` is a structure that maps typed keys to a set of typed values. <b>This typing is very important: just as the type-specific compiled code behind a NumPy array makes it more efficient than a Python list for certain operations, the type information of a Pandas `Series` makes it much more efficient than Python dictionaries for certain operations.</b>

We can make the Series-as-dictionary analogy even more clear by constructing a `Series` object directly from a Python dictionary:

In [21]:
population_dict = {'California': 38332521,
                   'Texas': 26448193,
                   'New York': 19651127,
                   'Florida': 19552860,
                   'Illinois': 12882135
                  }

population = pd.Series(population_dict)
population

California    38332521
Texas         26448193
New York      19651127
Florida       19552860
Illinois      12882135
dtype: int64

By default, a `Series` will be created where the index is drawn from the sorted keys. From here, a typical dictionary-style item access can be performed:

In [22]:
population['California']

38332521

Unlike a dictionary, however, the `Series` also supports array-style operations such as slicing:

In [23]:
population['California':'Illinois']

California    38332521
Texas         26448193
New York      19651127
Florida       19552860
Illinois      12882135
dtype: int64

<h3>Constructing Series Objects</h3>

Pandas `Series` creations are all some version of the following:
```python
>>> pd.Series(data, index=index)
```
where index is an optional argument, and data can be one of many entities.

For example, data can be a list or NumPy array, in which case index defaults to an integer sequence:

In [24]:
pd.Series([2, 4, 6])

0    2
1    4
2    6
dtype: int64

* data can be a scalar, which is repeated to fill the specified index:

In [25]:
pd.Series(5, index=[100, 200, 300])

100    5
200    5
300    5
dtype: int64

* data can be a dictionary, in which index defaults to the sorted dictionary keys:

In [26]:
pd.Series({2: 'a', 1: 'b', 3: 'c'})

2    a
1    b
3    c
dtype: object

* Note how in this case, the `Series` is only populated with the explicitly identified keys:

In [28]:
pd.Series({2: 'a', 1: 'b', 3: 'c'}, index=[3, 2])

3    c
2    a
dtype: object

<h3>The Pandas DataFrame Object</h3>

The next fundamental structure in Pandas is the `DataFrame`. Like the `Series` object discussed in the previous section, the `DataFrame` can be thought of either as a generalization of a NumPy array, or as a specialization of a Python dictionary. We'll now take a look at each of these perspectives.

<h4>DataFrame as a generalized NumPy array</h4>

If a `Series` is an analog of a one-dimensional array with flexible indices, a `DataFrame` is an analog of a two-dimensional array with both flexible row indices and flexible column names. Just as you might think of a two-dimensional array as an ordered sequence of aligned one-dimensional columns, <b>you can think of a `DataFrame` as a sequence of aligned `Series` objects. Here, by "aligned", we mean that they share the same index.</b>

To demonstrate this, let's first construct a new `Series` listing the area of each of the five states discussed in the previous section:

In [33]:
area_dict = {'California': 423967, 'Texas': 695662, 'New York': 141297,
             'Florida': 170312, 'Illinois': 149995}
area = pd.Series(area_dict)
area

California    423967
Texas         695662
New York      141297
Florida       170312
Illinois      149995
dtype: int64

* Now that we have this along with the `population Series` from before, we can use a dictionary to construct a single two-dimensional object containing this information:

In [34]:
states = pd.DataFrame({'population': population, 'area': area})
states

,population,area
California,38332521,423967
Texas,26448193,695662
New York,19651127,141297
Florida,19552860,170312
Illinois,12882135,149995


* Like the `Series` object, the `DataFrame` has an index attribute that gives access to the index labels. It also has a columns attribute:

In [35]:
states.index

Index(['California', 'Texas', 'New York', 'Florida', 'Illinois'], dtype='object')

In [36]:
states.columns

Index(['population', 'area'], dtype='object')

Thus the `DataFrame` can be thought of as a generalization of a two-dimensional NumPy array, where both the rows and columns have a generalized index for accessing the data.

<h3>DataFrame as a specialized dictionary</h3>

We can also think of a `DataFrame` as a specialization of a dictionary. Where a dictionary maps a key to a value, a `DataFrame` maps a column name to a `Series` of column data. For example, asking for the 'area' attribute returns the `Series` object containing the areas we saw earlier:

In [37]:
states['area']

California    423967
Texas         695662
New York      141297
Florida       170312
Illinois      149995
Name: area, dtype: int64

In [38]:
type(states['area'])

pandas.core.series.Series

Note the potential confusion here: in a two-dimensional NumPy array, `data[0]` will return the first <i>row</i>. For a `DataFrame`, `data['col0']` will return the first <i>column</i>. Because of this, it is probably better to think about `DataFrames` as generalized dictionaries rather than generalized arrays, though both ways of looking at the situation can be useful. We'll explore more flexible means of indexing `DataFrames` in a bit.

<h4>Constructing DataFrame objects</h4>

A Pandas `DataFrame` object can be constructed in a variety of ways. Here are several examples:

* <b>From a single `Series` object.</b> A `DataFrame` is a collection of `Series` objects, and a single-column `DataFrame` can be constructed from a single `Series`:

In [39]:
pd.DataFrame(population, columns=['population'])

,population
California,38332521
Texas,26448193
New York,19651127
Florida,19552860
Illinois,12882135


* <b>From a list of dicts</b>: Any list of dictionaries can be made into a `DataFrame`. We'll use a simple list comprehension to create some data:

In [40]:
data = [{'a': i, 'b': 2 * i} for i in range(3)]
data

[{'a': 0, 'b': 0}, {'a': 1, 'b': 2}, {'a': 2, 'b': 4}]

In [41]:
pd.DataFrame(data)

,a,b
0,0,0
1,1,2
2,2,4


* Even if some keys in the dictionary are missing, Pandas will fill them with `NaN` values:

In [42]:
pd.DataFrame([{'a': 1, 'b': 2}, {'b': 3, 'c': 4}])

,a,b,c
0,1.0,2,NaN
1,NaN,3,4.0


* <b>From a dictionary of Series objects</b>: A `DataFrame` can be constructed from a dictionary of `Series` objects as well:

In [43]:
pd.DataFrame({'population': population,
              'area': area})

,population,area
California,38332521,423967
Texas,26448193,695662
New York,19651127,141297
Florida,19552860,170312
Illinois,12882135,149995


* <b>From a two-dimensional NumPy array</b>: Given a two-dimensional array of data, we can create a `DataFrame` with any specified column and index names. If omitted, an integer index will be used for each:

In [44]:
pd.DataFrame(np.random.rand(3, 2),
             columns=['foo', 'bar'],
             index=['a', 'b', 'c']
            )

,foo,bar
a,0.165192,0.734091
b,0.633021,0.490757
c,0.316198,0.609857


* <b>From a NumPy structured array</b>: A Pandas `DataFrame` operates much like a structured array, and can be created directly from one:

In [46]:
A = np.zeros(3, dtype=[('A', 'i8'), ('B', 'f8')])
A

array([(0, 0.), (0, 0.), (0, 0.)], dtype=[('A', '<i8'), ('B', '<f8')])

In [47]:
pd.DataFrame(A)

,A,B
0,0,0.0
1,0,0.0
2,0,0.0


<h4>The Pandas Index Object</h4>

We have seen that both the `Series` and `DataFrame` objects contain an explicit <i>index</i> that lets you reference and modify data. This `Index` object is an interesting structure in itself, and can be thought of either as an <i>immutable array</i> or as an <i>ordered set</i> (technically a multiset, as `Index` objects may contain repeated values). Let's construct an `Index` from a list of integers to see its operations:

In [48]:
ind = pd.Index([2, 3, 5, 7, 11])
ind

Int64Index([2, 3, 5, 7, 11], dtype='int64')

* <b>Index as immutable array</b>:

The `Index` object in many ways operates like an array. For example, we can use standard Python indexing notation to retrieve values or slices:

In [49]:
ind[1]

3

In [50]:
ind[::2]

Int64Index([2, 5, 11], dtype='int64')

* Index objects also have many of the attributes familiar from NumPy arrays:

In [51]:
print(ind.size, ind.shape, ind.ndim, ind.dtype)

5 (5,) 1 int64


* One key difference between `Index` objects and NumPy arrays is that indices are immutable -- that is, they cannot be modified via the normal means. This immutability makes it safer to share indices between multiple `DataFrames` and arrays, without the potential for side effects from inadvertent index modification:

In [52]:
ind[1] = 0

TypeError: Index does not support mutable operations

* <b>Index as ordered set</b>:

Pandas objects are designed to facilitate operations such as joins across datasets, which depend on many aspects of set arithmetic. The `Index` object follows many of the conventions used by Python's built-in `set` data structure, so that unions, intersections, differences, and other combinations can be computed in a familiar way:

In [53]:
indA = pd.Index([1, 3, 5, 7, 9])
indB = pd.Index([2, 3, 5, 7, 11])

In [54]:
indA & indB # intersection

<ipython-input-54-f400ec9c4e08>:1: FutureWarning: Index.__and__ operating as a set operation is deprecated, in the future this will be a logical operation matching Series.__and__.  Use index.intersection(other) instead
  indA & indB # intersection


Int64Index([3, 5, 7], dtype='int64')

In [56]:
indA | indB # union

<ipython-input-56-0a17b8447828>:1: FutureWarning: Index.__or__ operating as a set operation is deprecated, in the future this will be a logical operation matching Series.__or__.  Use index.union(other) instead
  indA | indB # union


Int64Index([1, 2, 3, 5, 7, 9, 11], dtype='int64')

In [57]:
indA ^ indB # symmetric difference

<ipython-input-57-aebc1839486a>:1: FutureWarning: Index.__xor__ operating as a set operation is deprecated, in the future this will be a logical operation matching Series.__xor__.  Use index.symmetric_difference(other) instead
  indA ^ indB # symmetric difference


Int64Index([1, 2, 9, 11], dtype='int64')

* These operations can also be accessed via object methods -- e.g.,

```python
indA.intersection(indB)
```

which is also the recommendation in those warning messages.

<h3>Data Indexing and Selection</h3>

* Indexing, slicing, masking, fancy indexing, etc. can be used for NumPy arrays. Here we'll explore similar means of accessing and modifying values in Pandas `Series` and `DataFrame` objects. Let's start with one-dimensional `Series` object, and then move on to the more complicated two-dimensional `DataFrame` object.

<h4>Data Selection in Series</h4>

* As we saw earlier, a `Series` object acts in many ways like a one-dimensional NumPy array, and in many ways like a standard Pythonic dictionary. Keep these two in mind when understanding the patterns of data indexing and selection in these arrays.

* <b>Series as dictionary</b>: mapping keys to collection of values

In [59]:
data = pd.Series([0.25, 0.5, 0.75, 1.0], index=['a', 'b', 'c', 'd'])
data

a    0.25
b    0.50
c    0.75
d    1.00
dtype: float64

In [60]:
'a' in data

True

In [61]:
data.keys()

Index(['a', 'b', 'c', 'd'], dtype='object')

In [62]:
list(data.items())

[('a', 0.25), ('b', 0.5), ('c', 0.75), ('d', 1.0)]

In [63]:
data['e'] = 1.25
data

a    0.25
b    0.50
c    0.75
d    1.00
e    1.25
dtype: float64

* <b>Series as one-dimensional array</b>:

A series builds on this dictionary-like interface and provides array-style item selection via the same basic mechanisms as NumPy arrays -- that is, <i>slices, masking, and fancy indexing</i>. Examples:

In [66]:
# slicing by explicit index
data['a':'c']

a    0.25
b    0.50
c    0.75
dtype: float64

In [67]:
# slicing by implicit integer index
data[0:2]

a    0.25
b    0.50
dtype: float64

In [68]:
# masking
data[(data > 0.3) & (data < 0.8)]

b    0.50
c    0.75
dtype: float64

In [69]:
# fancy indexing
data[['a', 'e']]

a    0.25
e    1.25
dtype: float64

* Among these, slicing may be the most confusing -- notice that when you slice with explicit index, the final index is <i>included</i> in the slice, while when you slice with the implicit integer index, the final index is <i>excluded</i> from the slice.

<h4>Indexers: loc, iloc, and ix</h4>

* These slicing and indexing conventions can be a source of confusion. For example, if your `Series` has an explicit integer index, an indexing operation such as `data[1]` will use the explicit indices, while a slicing operation like `data[1:3]` will use the implicit Python-style index. Example

In [110]:
data = pd.Series(['a', 'b', 'c'], index=[1, 3, 5])
data

1    a
3    b
5    c
dtype: object

In [111]:
# explicit index when indexing
data[1]

'a'

In [112]:
# implicit index when slicing
data[1:3]

3    b
5    c
dtype: object

* Due to the potential confusion in the case of integer indexes, Pandas provides some special <i>indexer</i> attributes that explicitly expose certain indexing schemes. These are not functional methods, but attributes that expose a particular slicing interface to the data in the `Series`.

* First, the `loc` attribute allows indexing and slicing that always references the explicit index:

In [74]:
data.loc[3]

'b'

In [76]:
data.loc[1:4]

1    a
3    b
dtype: object

* The `iloc` attribute allows indexing and slicing that always references the implict, Python-style index:

In [114]:
data.iloc[0]

'a'

In [113]:
data.iloc[1]

'b'

In [81]:
data.iloc[1:4]

3    b
5    c
dtype: object

* The `ix` indexing attribute is a hybrid of the two, and will become more apparent in the context of `DataFrame` objects. (It has actually been deprecated)

* <b>One guiding principle of Python is that "explicit is better than implicit". The explicit nature of `loc` and `iloc` make them very useful in maintaining clean and readable code; especially in the case of integer indexes. It is recommended to use both of these to make code easier to read and understand, and to prevent subtle bugs due to the mixed indexing / slicing convention.</b>

<h4>Data Selection in DataFrame</h4>

* <b>DataFrame as dictionary</b>:

In [115]:
area = pd.Series({'California': 423967, 'Texas': 695662,
                  'New York': 141297, 'Florida': 170312,
                  'Illinois': 149995
                 }
                )

pop = pd.Series({'California': 38332521, 'Texas': 26448193,
                 'New York': 19651127, 'Florida': 19552860,
                 'Illinois': 12882135
                }
               )
data = pd.DataFrame({'area': area, 'pop': pop})
data

,area,pop
California,423967,38332521
Texas,695662,26448193
New York,141297,19651127
Florida,170312,19552860
Illinois,149995,12882135


* The individual `Series` that made up the columns of the `DataFrame` can be accessed via dictionary-style indexing of the column name:

In [116]:
data['area']

California    423967
Texas         695662
New York      141297
Florida       170312
Illinois      149995
Name: area, dtype: int64

* Equivalently, we can also use attribute-style access with column names that are strings:

In [117]:
data.area

California    423967
Texas         695662
New York      141297
Florida       170312
Illinois      149995
Name: area, dtype: int64

* Note that this attribute-style column access actually accesses the same exact object as the dictionary-style access:

In [118]:
data.area is data['area']

True

* Note that if the column names are not strings, this will not work. Also, if the column names conflict with methods of the `DataFrame`, attribute-style access is not possible. For example, the `DataFrame` has a `pop()` method, so data.pop will point to this rather than the "pop" column:

In [119]:
data.pop is data['pop']

False

* In general, avoid the temptation to try column assignment via attribute (i.e., use `data['pop']` = z rather than `data.pop = z`).

In [120]:
data['density'] = data['pop'] / data['area']
data

,area,pop,density
California,423967,38332521,90.413926
Texas,695662,26448193,38.018740
New York,141297,19651127,139.076746
Florida,170312,19552860,114.806121
Illinois,149995,12882135,85.883763


* <b>DataFrame as two-dimensional array</b>

In [121]:
data.values

array([[4.23967000e+05, 3.83325210e+07, 9.04139261e+01],
       [6.95662000e+05, 2.64481930e+07, 3.80187404e+01],
       [1.41297000e+05, 1.96511270e+07, 1.39076746e+02],
       [1.70312000e+05, 1.95528600e+07, 1.14806121e+02],
       [1.49995000e+05, 1.28821350e+07, 8.58837628e+01]])

In [125]:
# transposte the full df to swap rows and columns
data.T

,California,Texas,New York,Florida,Illinois
area,4.239670e+05,6.956620e+05,1.412970e+05,1.703120e+05,1.499950e+05
pop,3.833252e+07,2.644819e+07,1.965113e+07,1.955286e+07,1.288214e+07
density,9.041393e+01,3.801874e+01,1.390767e+02,1.148061e+02,8.588376e+01


* When it comes to indexing `DataFrame` objects, the dictionary-style indexing of columns precludes our ability to simply treat it as a NumPy array. Passing a single index to an array accesses a row:

In [124]:
data.values[0]

array([4.23967000e+05, 3.83325210e+07, 9.04139261e+01])

In [126]:
# passing a single "index" to a df accesses a column
data['area']

California    423967
Texas         695662
New York      141297
Florida       170312
Illinois      149995
Name: area, dtype: int64

* We can again use the `loc` and `iloc` indexers mentioned earlier. Using the `iloc` indexer, we can index the underlying array as if it is a simply NumPy array (using implict Python-style index), but the `DataFrame` index and column labels are maintained in the result:

In [128]:
# first 3 rows, first 2 columns
data.iloc[:3, :2]

,area,pop
California,423967,38332521
Texas,695662,26448193
New York,141297,19651127


In [131]:
# all rows up to and including 'Illinois', columns up to and including 'pop'
data.loc[:'Illinois', :'pop']

,area,pop
California,423967,38332521
Texas,695662,26448193
New York,141297,19651127
Florida,170312,19552860
Illinois,149995,12882135


In [134]:
data.loc[data.density > 100, ['pop', 'density']]

,pop,density
New York,19651127,139.076746
Florida,19552860,114.806121


In [135]:
data.iloc[0, 2] = 90
data

,area,pop,density
California,423967,38332521,90.000000
Texas,695662,26448193,38.018740
New York,141297,19651127,139.076746
Florida,170312,19552860,114.806121
Illinois,149995,12882135,85.883763


In [136]:
data['Florida':'Illinois']

,area,pop,density
Florida,170312,19552860,114.806121
Illinois,149995,12882135,85.883763


In [139]:
data[3:5]

,area,pop,density
Florida,170312,19552860,114.806121
Illinois,149995,12882135,85.883763


In [140]:
data[data.density > 100]

,area,pop,density
New York,141297,19651127,139.076746
Florida,170312,19552860,114.806121


In [141]:
data[data['density'] > 100]

,area,pop,density
New York,141297,19651127,139.076746
Florida,170312,19552860,114.806121


<h3>Operating on Data in Pandas</h3>

* Pandas inherits much of its functionality from NumPy and the ufuncs on pg. 50. It can perform quick element-wise operations, both with basic arithmetic (addition, subtraction, multiplication, etc.), and more sophisticated functions (trigonometric functions, exponential and logarithmic functions, etc.).

In [146]:
rng = np.random.RandomState(42)
ser = pd.Series(rng.randint(0, 10, 4))
ser

0    6
1    3
2    7
3    4
dtype: int64

In [147]:
df = pd.DataFrame(rng.randint(0, 10, (3, 4)),
                  columns=['A', 'B', 'C', 'D'])
df

,A,B,C,D
0,6,9,2,6
1,7,4,3,7
2,7,2,5,4


In [196]:
np.exp(ser)

0     403.428793
1      20.085537
2    1096.633158
3      54.598150
dtype: float64

In [197]:
np.sin(df * np.pi / 4)

,A,B,C,D
0,-1.000000,7.071068e-01,1.000000,-1.000000e+00
1,-0.707107,1.224647e-16,0.707107,-7.071068e-01
2,-0.707107,1.000000e+00,-0.707107,1.224647e-16


In [198]:
area = pd.Series({'Alaska': 1723337, 'Texas': 695662,
                  'California': 423967}, name='area')
population = pd.Series({'California': 38332521, 'Texas': 26448193,
                        'New York': 19651127}, name='population')

In [199]:
population

California    38332521
Texas         26448193
New York      19651127
Name: population, dtype: int64

In [200]:
area

Alaska        1723337
Texas          695662
California     423967
Name: area, dtype: int64

In [201]:
population / area

Alaska              NaN
California    90.413926
New York            NaN
Texas         38.018740
dtype: float64

* The resulting array contains the <i>union</i> of indices of the two input arrays, which we could determine using standard Python set arithmetic on these indices. Any item for which one or the other does not have an entry is marked with `NaN` since it is missing. 

In [206]:
area.index | population.index

<ipython-input-206-ff558a211efb>:1: FutureWarning: Index.__or__ operating as a set operation is deprecated, in the future this will be a logical operation matching Series.__or__.  Use index.union(other) instead
  area.index | population.index


Index(['Alaska', 'California', 'New York', 'Texas'], dtype='object')

In [207]:
A = pd.Series([2, 4, 6], index=[0, 1, 2])
B = pd.Series([1, 3, 5], index=[1, 2, 3])
A + B

0    NaN
1    5.0
2    9.0
3    NaN
dtype: float64

* If using `NaN` values is not desired behavior, we can modify the fill value using appropriate object methods in place of the operators. Using `A.add(B)` is equivalent to `A + B`, but allows optional explicit specification of the fill value for any elements in `A` or `B` that might be missing:

In [208]:
A.add(B, fill_value=0)

0    2.0
1    5.0
2    9.0
3    5.0
dtype: float64

* This Index alignment happens in both columns and indices when you are performing operations on `DataFrames`

In [209]:
A = pd.DataFrame(rng.randint(0, 20, (2, 2)),
                 columns=list('AB'))
B = pd.DataFrame(rng.randint(0, 10, (3, 3)),
                 columns=list('BAC'))

In [210]:
A

,A,B
0,1,11
1,5,1


In [211]:
B

,B,A,C
0,4,0,9
1,5,8,0
2,9,2,6


In [212]:
A + B

,A,B,C
0,1.0,15.0,NaN
1,13.0,6.0,NaN
2,NaN,NaN,NaN


* Note how the indices are aligned correctly irrespective of their order in the two objects, and indices in the result are sorted. We can also pass any `fill_value` to be used in place of missing entries. Here we'll fill the mean of all values in `A` (which we compute by first stacking the rows of `A`):

In [218]:
# this gives us a single value of 4.5
fill = A.stack().mean()

In [221]:
A.add(B, fill_value=fill)

,A,B,C
0,1.0,15.0,13.5
1,13.0,6.0,4.5
2,6.5,13.5,10.5


In [222]:
A = rng.randint(10, size=(3, 4))
A

array([[3, 8, 2, 4],
       [2, 6, 4, 8],
       [6, 1, 3, 8]])

In [223]:
# difference between two-dimensional array and one of its rows
A - A[0]

array([[ 0,  0,  0,  0],
       [-1, -2,  2,  4],
       [ 3, -7,  1,  4]])

* According to NumPy's broadcasting rules, subtraction between a two-dimensional array and one of its rows is applied row-wise. In Pandas, the convention similarly operates row-wise by default:

In [224]:
df = pd.DataFrame(A, columns=list('QRST'))
df

,Q,R,S,T
0,3,8,2,4
1,2,6,4,8
2,6,1,3,8


In [225]:
df - df.iloc[0]

,Q,R,S,T
0,0,0,0,0
1,-1,-2,2,4
2,3,-7,1,4


In [226]:
# In order to do this column-wise, you can use the axis keyword with an object method
df.subtract(df['R'], axis=0)

,Q,R,S,T
0,-5,0,-6,-4
1,-4,0,-2,2
2,5,0,2,7


In [227]:
halfrow = df.iloc[0, ::2]
halfrow

Q    3
S    2
Name: 0, dtype: int64

In [228]:
df - halfrow

,Q,R,S,T
0,0.0,NaN,0.0,NaN
1,-1.0,NaN,2.0,NaN
2,3.0,NaN,1.0,NaN


<h3>Handling Missing Data</h3>

Generally, handling missing data revolves around one of two strategies: using a <i>mask</i> that globally indicates missing values, or choosing a <i>sentinel value</i> that indicates a missing entry.

For the mask approach, it might be an entirely separate Boolean array, or it may involve using one bit of the data representation locally to indicate the null status of a value.

The sentinel value approach may use some data-specific convention, such as a missing int with -9999.

Tradeoffs: using a separate mask array requires allocation of the additional Boolean array, adding overhead to both storage and computation. A sentinel value reduces the range of valid values, and may require extra logic in CPU / GPU arithmetic. Common special values like `NaN` are not available for all data types.

Pandas handles missing values by using the NumPy package, which does not have built-in notion of NA values for non-floating-point data types. Pandas could have followed R's lead in specifying bit patterns for each individual data type to indicate nullness, but this approach is unwieldy. R contains only 4 basic data types, while NumPy supports <i>far</i> more than this: R has only a single integer type, NumPy has 14 basic integer types. 

NumPy does have support for masked arrays -- that is, arrays that have a separate Boolean mask array attached for marking data as "good" or "bad". Pandas could have derived from this, but the overhead in both storage, computation, and code maintenance makes that an unattractive choice.

<b>Pandas chose to use sentinels for missing data, and further chose to use two already-existing Python null values: the special floating-point `NaN` value, and the Python `None` object. This choice has some side effects, as we will see, but in practice ends up being a good compromise in most cases of interest.</b>

* The first sentinel value by Pandas is `None`, a Python singleton object that is often used for missing data in Python code. Because `None` is a Python object, it cannot be used in any arbitrary NumPy/Pandas array, but only in arrays with data type `'object'` (i.e., arrays of Python objects):

In [229]:
vals1 = np.array([1, None, 3, 4])
vals1

array([1, None, 3, 4], dtype=object)

<h3>**Drastic Performance Differences**</h3>

* <b>This `dtype=object` means that the best common type representation NumPy could infer for the contents of the array is that they are Python objects. While this kind of object array is useful for some purposes, any operations on the data will be done at the Python level, with much more overhead than the typically fast operators seen for arrays with native types. Example below showing a dramatic performance difference:</b>

In [230]:
for dtype in ['object', 'int']:
    print("dtype =", dtype)
    %timeit np.arange(1E6, dtype=dtype).sum()
    print()

dtype = object
41.3 ms ± 391 µs per loop (mean ± std. dev. of 7 runs, 10 loops each)

dtype = int
579 µs ± 4.41 µs per loop (mean ± std. dev. of 7 runs, 1000 loops each)



* The use of Python objects in an array also means that if you perform aggregations like `sum()` or `min()` across an array with a `None` value, you will generally get an error. This is due to the fact that addition between an integer and `None` is undefined:

In [239]:
vals1.sum()

TypeError: unsupported operand type(s) for +: 'int' and 'NoneType'

* `NaN` for missing numerical data:

NumPy chose a native floating-point type for this array: this means that unlike the object array from before, this array supports fast operations pushed into compiled code. You should be aware that `NaN` is a bit like a data virus -- it infects any other object it touches. Regardless of the operation, the result of arithmetic with `NaN` will be another `NaN`:

In [242]:
vals2 = np.array([1, np.nan, 3, 4])
vals2.dtype

dtype('float64')

In [243]:
1 + np.nan

nan

In [244]:
0 * np.nan

nan

* Since `NaN` "infects" the other numbers, this means that aggregates over the values don't result in an error, but they are not useful:

In [245]:
vals2.sum(), vals2.min(), vals2.max()

(nan, nan, nan)

* There are special aggregations that will ignore these missing values:

In [246]:
np.nansum(vals2), np.nanmin(vals2), np.nanmax(vals2)

(8.0, 1.0, 4.0)

* Keep in mind that `NaN` is specifically a floating-point value; there is no equivalent `NaN` value for integers, strings, or other types.

* `NaN` and `None` both have their place, and Pandas is built to handle the two of them nearly interchangeably, converting between them when appropriate:

In [249]:
pd.Series([1, np.nan, 2, None])

0    1.0
1    NaN
2    2.0
3    NaN
dtype: float64

* For types that don't have an available sentinel value, Pandas automatically type-casts when NA values are present. For example, if we set a value in an integer array to `np.nan`, it will automatically be upcast to a floating-point type to accomadate the NA:

In [250]:
x = pd.Series(range(2), dtype=int)
x

0    0
1    1
dtype: int64

In [252]:
x[0] = None
x

0    NaN
1    1.0
dtype: float64

* Notice that Pandas not only casted the integer array to floating point, but also automatically converted the `None` to a `NaN` value. (Native integer NA may be added to Pandas in the future, but has not been yet).

| Typeclass   | Conversion When Storing NAs               | NA Sentinel Value   |
|------------|-------------------------------------------|---------------------|
| Floating   | Remains `float64`                        | `np.nan`           |
| Object     | Remains `object`                         | `None` or `np.nan` |
| Integer    | Casted to `float64`                      | `np.nan`           |
| Boolean    | Casted to `object`                       | `None` or `np.nan` |

* Note that in Pandas, string data is always stored with an `object` dtype.

<h4>Operating on Null Values</h4>

* As we have seen, Pandas treats `NaN` and `None` as essentially interchangeable for indicating missing or null values. To facilitate this convention, there are several useful methods for detecting, removing, and replacing null values in Pandas data structures. These are:

* `isnull()` : Generate a Boolean mask indicating missing values
* `notnull()` : Opposite of `isnull()`
* `dropna()` : Return a filtered version of the data, with nulls dropped
* `fillna()` : Return a copy of the data with missing values filled or imputed


In [253]:
data = pd.Series([1, np.nan, 'hello', None])
data

0        1
1      NaN
2    hello
3     None
dtype: object

In [254]:
data.isnull()

0    False
1     True
2    False
3     True
dtype: bool

In [257]:
data[data.notnull()]

0        1
2    hello
dtype: object

In [258]:
data[data.isnull()]

1     NaN
3    None
dtype: object

* The `isnull()` and `notnull()` methods produce similar Boolean results for `DataFrames`; they are opposites of each other.

* <b>Dropping null values</b>

In addition to the masking used before, there are convenience methods, `dropna()` (which removes NA values), and `fillna()` (which fills NA values). For a `Series`, the result is straightforward:

In [259]:
data

0        1
1      NaN
2    hello
3     None
dtype: object

In [262]:
# note that this does not change 'data' in-place
data.dropna()

0        1
2    hello
dtype: object

* For a `DataFrame`, there are more options. We cannot drop single values from a `DataFrame`; you must only drop full rows or full columns, so `dropna()` gives a number of options for a `DataFrame`. <b>By default, `dropna()` drops all rows in which <i>any</i> null value is present:</b>

In [265]:
df = pd.DataFrame([[1,      np.nan, 2],
                   [2,      3,      5],
                   [np.nan, 4,      6]])
df

,0,1,2
0,1.0,NaN,2
1,2.0,3.0,5
2,NaN,4.0,6


In [266]:
df.dropna()

,0,1,2
1,2.0,3.0,5


* We can drop NA values along a different axis; `axis=1` drops all columns containing a null value:

In [267]:
df.dropna(axis=1) # or axis='columns'

,2
0,2
1,5
2,6


* Note that this drops some good data as well; we might rather be interested in dropping rows or columns with <i>all</i> NA values, or a majority of NA values. This can be specified through the `how` or `thresh` parameters, which allow fine control of the number of nulls to allow through.
* The default is `how='any'`, such that any row or column (depending on the `axis` keyword) containing a null value will be dropped. You can also specify `how='all'`, which will only drop rows/columns that are <i>all</i> null values:

In [268]:
df[3] = np.nan
df

,0,1,2,3
0,1.0,NaN,2,NaN
1,2.0,3.0,5,NaN
2,NaN,4.0,6,NaN


In [270]:
# this drops column 3 since all values are NaN
df.dropna(axis='columns', how='all')

,0,1,2
0,1.0,NaN,2
1,2.0,3.0,5
2,NaN,4.0,6


* For finer-grained control, the `thresh` parameter lets you specify a minimum number of non-null values for the rows/column to be kept:

In [272]:
# drop first and last rows, because they contain only 2 non-null values
df.dropna(axis='rows', thresh=3)

,0,1,2,3
1,2.0,3.0,5,NaN


* <b>Filling null values</b>

Sometimes rather than dropping NA values, you want to replace them with a valid value. This might be a single number like zero, or some imputation or interpolation from the good values. The `fillna()` method returns a copy of the array with the null values replaced:

In [273]:
data = pd.Series([1, np.nan, 2, None, 3], index=list('abcde'))
data

a    1.0
b    NaN
c    2.0
d    NaN
e    3.0
dtype: float64

In [274]:
# fill NA entries with single value of 0
data.fillna(0)

a    1.0
b    0.0
c    2.0
d    0.0
e    3.0
dtype: float64

In [275]:
# forward-fill to propagate the previous value forward
data.fillna(method='ffill')

a    1.0
b    1.0
c    2.0
d    2.0
e    3.0
dtype: float64

In [276]:
# back-fill to propagate the next values backward:
data.fillna(method='bfill')

a    1.0
b    2.0
c    2.0
d    3.0
e    3.0
dtype: float64

* The methods are similar in `DataFrames`, but we can also specify an `axis` along which the fills take place:

In [277]:
df

,0,1,2,3
0,1.0,NaN,2,NaN
1,2.0,3.0,5,NaN
2,NaN,4.0,6,NaN


In [278]:
df.fillna(method='ffill', axis=1)

,0,1,2,3
0,1.0,1.0,2.0,2.0
1,2.0,3.0,5.0,5.0
2,NaN,4.0,6.0,6.0


* Note that if a previous value is not available during a forward fill, the NA value remains. Same with a back fill.

In [280]:
df.fillna(method='bfill', axis=1)

,0,1,2,3
0,1.0,2.0,2.0,NaN
1,2.0,3.0,5.0,NaN
2,4.0,4.0,6.0,NaN


<h3>Hierarchical Indexing</h3>

* We've focused primarily on one-dimensional and two-dimensional data so far, stored in Pandas `Series` and `DataFrame` objects. It's often useful to go beyond this and store higher-dimensional data -- data indexed by more than one or two keys. Pandas does provide `Panel` and `Panel4D` objects that natively handle three-dimensional and four-dimensional data, but a far more common practice is to use <i>hierarchical indexing</i> (also called multi-indexing) to incorporate index levels within a single index. In this way, higher-dimensional data can be compactly represented within the familiar one-dimensional `Series` and two-dimensional `DataFrame` objects.

* We'll explore the `MultiIndex` objects; considerations around indexing, slicing, and computing statistics across multiply indexed data; and useful routines for converting between simple and hierarchically indexed representations of your data.

* <b>A Multiply Indexed Series</b>

Let's start by considering how we might represent two-dimensional data within a one-dimensional `Series`. Let's consider a series of data where each point has a character and numerical key.

* Suppose you want to track data about states from two different years. Using the Pandas tools we've already covered, you might be tempted to simply use Python tuples as keys.

* The bad way:

In [281]:
index = [('California', 2000), ('California', 2010),
         ('New York', 2000), ('New York', 2010),
         ('Texas', 2000), ('Texas', 2010)]

populations = [33871648, 37253956,
              18976457, 19378102,
              20851820, 25145561]
pop = pd.Series(populations, index=index)
pop

(California, 2000)    33871648
(California, 2010)    37253956
(New York, 2000)      18976457
(New York, 2010)      19378102
(Texas, 2000)         20851820
(Texas, 2010)         25145561
dtype: int64

In [282]:
# we can straightforwardly index or slice the series based on this multiple index
pop[('California', 2010):('Texas', 2000)]

(California, 2010)    37253956
(New York, 2000)      18976457
(New York, 2010)      19378102
(Texas, 2000)         20851820
dtype: int64

* However, the convenience ends here. If we need to select all values from 2010, then we need to do some messy (potentially inefficient) data munging to make it happen:

In [283]:
pop[[i for i in pop.index if i[1] == 2010]]

(California, 2010)    37253956
(New York, 2010)      19378102
(Texas, 2010)         25145561
dtype: int64

* <b>The better way : Pandas MultiIndex</b>

Pandas provides a better way. The tuple-based indexing is essentially a rudimentary multi-index, and the Pandas `MultiIndex` type gives us the type of operations we wish to have. We can create a multi-index from the tuples as follows:

In [287]:
index = pd.MultiIndex.from_tuples(index)
index

MultiIndex([('California', 2000),
            ('California', 2010),
            (  'New York', 2000),
            (  'New York', 2010),
            (     'Texas', 2000),
            (     'Texas', 2010)],
           )

* If we reindex our series with this `MultiIndex`, we see the hierarchical representation of the data.
* Here, the first 2 columns of the `Series` representation show the multiple index values, while the third column shows the data. Notice that some entries are missing in the first column: in this multi-index representation, any blank entry indicates the same value as the line above it.

In [289]:
pop = pop.reindex(index)
pop

California  2000    33871648
            2010    37253956
New York    2000    18976457
            2010    19378102
Texas       2000    20851820
            2010    25145561
dtype: int64

In [290]:
# now to access all data for which the second index is 2010, we simply use Pandas slicing
pop[:, 2010]

California    37253956
New York      19378102
Texas         25145561
dtype: int64

* Now, the syntax is much more convenient, and the operation is much more efficient than the custom tuple-based multi-index that we started with. 

<h3>Multi-Index as extra dimension</h3>

* We could have easily stored this same data using a simple `DataFrame` with index and column labels. Pandas is built with this equivalence in mind. The `unstack()` method will quickly convert a multiply-indexed `Series` into a conventionally indexed `DataFrame`, and the `stack()` method will do the opposite (`DataFrame` to multiply-indexed `Series`). Example below:

In [292]:
pop

California  2000    33871648
            2010    37253956
New York    2000    18976457
            2010    19378102
Texas       2000    20851820
            2010    25145561
dtype: int64

In [291]:
pop_df = pop.unstack()
pop_df

,2000,2010
California,33871648,37253956
New York,18976457,19378102
Texas,20851820,25145561


In [293]:
pop_df.stack()

California  2000    33871648
            2010    37253956
New York    2000    18976457
            2010    19378102
Texas       2000    20851820
            2010    25145561
dtype: int64

* <b>So why even bother with multi-indexing at all?</b> The reason is simple: just as we were able to use multi-indexing to represent two-dimensional data within a one-dimensional `Series`, we can also use it to represent data of three or more dimensions in a `Series` or `DataFrame`. Each extra level in a multi-index represents an extra dimension of data; taking advantage of this property gives us much more flexibility in the types of data we can represent. Let's add another column of demographic data for each state at each year (population under 18) with a `MultiIndex`, which is as easy as adding another column to the `DataFrame`
* Another reason is that all of the ufuncs work with hierarchical indices as well.

In [294]:
pop_df = pd.DataFrame({'total': pop,
                       'under18': [9267089, 9284094,
                                   4687374, 4318033,
                                   5906301, 6879014]})
pop_df

total  under18
California 2000  33871648  9267089
           2010  37253956  9284094
New York   2000  18976457  4687374
           2010  19378102  4318033
Texas      2000  20851820  5906301
           2010  25145561  6879014

In [295]:
f_u18 = pop_df['under18'] / pop_df['total']
f_u18.unstack()

,2000,2010
California,0.273594,0.249211
New York,0.247010,0.222831
Texas,0.283251,0.273568


<h3>Methods of MultiIndex Creation</h3>

* The most straightforward way to construct a multiply indexed `Series` or `DataFrame` is to simply pass a list of two or more index arrays to the constructor. Example:

In [298]:
df = pd.DataFrame(np.random.rand(4, 2),
                  index=[['a', 'a', 'b', 'b'], [1, 2, 1, 2]],
                  columns=['data1', 'data2'])
df

data1     data2
a 1  0.837641  0.662627
  2  0.987701  0.970969
b 1  0.895907  0.208794
  2  0.545314  0.908795

* The work of creating the `MultiIndex` is done in the background
* If you pass a dictionary with appropriate tuples as keys, Pandas will automatically recognize this and use a `MultiIndex` by default:

In [299]:
data = {('California', 2000): 33871648,
        ('California', 2010): 37253956,
        ('Texas', 2000): 20851820,
        ('Texas', 2010): 25145561,
        ('New York', 2000): 18976457,
        ('New York', 2010): 19378102}

pd.Series(data)

California  2000    33871648
            2010    37253956
Texas       2000    20851820
            2010    25145561
New York    2000    18976457
            2010    19378102
dtype: int64

* It can sometimes be useful to explicitly create a `MultiIndex`, and we'll go through a couple methods here:

In [300]:
pd.MultiIndex.from_arrays([['a', 'a', 'b', 'b'], [1, 2, 1, 2]])

MultiIndex([('a', 1),
            ('a', 2),
            ('b', 1),
            ('b', 2)],
           )

In [301]:
pd.MultiIndex.from_tuples([('a', 1), ('a', 2), ('b', 1), ('b', 2)])

MultiIndex([('a', 1),
            ('a', 2),
            ('b', 1),
            ('b', 2)],
           )

In [303]:
# Cartesian product of single indices
pd.MultiIndex.from_product([['a', 'b'], [1, 2]])

MultiIndex([('a', 1),
            ('a', 2),
            ('b', 1),
            ('b', 2)],
           )

<b>MultiIndex level names</b>

* It can be useful to name the levels of the `MultiIndex`. With more involved datasets, this can be a useful way to keep track of the meanings of various index values. Pass the `names` arg to any of the above `MultiIndex` constructors, or by setting the `names` attribute of the index after the fact:

In [304]:
pop

California  2000    33871648
            2010    37253956
New York    2000    18976457
            2010    19378102
Texas       2000    20851820
            2010    25145561
dtype: int64

In [305]:
pop.index.names = ['state', 'year']
pop

state       year
California  2000    33871648
            2010    37253956
New York    2000    18976457
            2010    19378102
Texas       2000    20851820
            2010    25145561
dtype: int64

In [306]:
# p 133 

* In a `DataFrame`, just as the rows can have multiple levels of indices, the columns can have multiple levels as well.

In [307]:
index = pd.MultiIndex.from_product([[2013, 2014], [1, 2]],
                                   names=['year', 'visit'])

columns = pd.MultiIndex.from_product([['Bob', 'Guido', 'Sue'], ['HR', 'Temp']],
                                     names=['subject', 'type'])

# mock some data
data = np.round(np.random.randn(4, 6), 1)
data[:, ::2] *= 10
data += 37

In [308]:
health_data = pd.DataFrame(data, index=index, columns=columns)
health_data

subject      Bob       Guido         Sue      
type          HR  Temp    HR  Temp    HR  Temp
year visit                                    
2013 1      25.0  36.9  49.0  37.3  41.0  34.7
     2      27.0  36.5  33.0  37.6  37.0  35.2
2014 1      24.0  38.1  54.0  34.7  35.0  38.4
     2      52.0  36.1  40.0  36.1  40.0  37.4

* <b>Here we see why multi-indexing for both rows and columns can come in very handy. This is fundamentally four-dimensional data, where the dimensions are the subject, measurement type, the year, and the visit number.</b>

* With this in place, we can index the top-level column by the person's name and get a full `DataFrame` containing just that person's information.

In [312]:
health_data['Guido']

type          HR  Temp
year visit            
2013 1      49.0  37.3
     2      33.0  37.6
2014 1      54.0  34.7
     2      40.0  36.1

* For complicated records containing multiple labeled measurements across multiple times for many subjects (people, countries, cities, etc.), use of hierarchical rows and columns can be extremely convenient.

<h3>Indexing and Slicing a MultiIndex</h3>

In [313]:
pop

state       year
California  2000    33871648
            2010    37253956
New York    2000    18976457
            2010    19378102
Texas       2000    20851820
            2010    25145561
dtype: int64

In [314]:
pop['California', 2000]

33871648

In [315]:
# can also do "partial indexing", by indexing just one of the levels in the index
pop['California']

year
2000    33871648
2010    37253956
dtype: int64

In [316]:
# partial slicing
pop.loc['California':'New York']

state       year
California  2000    33871648
            2010    37253956
New York    2000    18976457
            2010    19378102
dtype: int64

In [317]:
# If index is sorted, we can perform partial indexing on lower levels
# by passing an empty slice in the first index:
pop[:, 2000]

state
California    33871648
New York      18976457
Texas         20851820
dtype: int64

In [318]:
# Boolean masks also work
pop[pop > 22_000_000]

state       year
California  2000    33871648
            2010    37253956
Texas       2010    25145561
dtype: int64

In [320]:
# fancy indexing (passing an array of indices):
pop[['California', 'Texas']]

state       year
California  2000    33871648
            2010    37253956
Texas       2000    20851820
            2010    25145561
dtype: int64

* <b>Multiply Indexed DataFrames</b>

These behave in a similar manner. Consider the medical data from before:

In [321]:
health_data

subject      Bob       Guido         Sue      
type          HR  Temp    HR  Temp    HR  Temp
year visit                                    
2013 1      25.0  36.9  49.0  37.3  41.0  34.7
     2      27.0  36.5  33.0  37.6  37.0  35.2
2014 1      24.0  38.1  54.0  34.7  35.0  38.4
     2      52.0  36.1  40.0  36.1  40.0  37.4

In [322]:
# view Guido's heart rate data
health_data['Guido', 'HR']

year  visit
2013  1        49.0
      2        33.0
2014  1        54.0
      2        40.0
Name: (Guido, HR), dtype: float64

In [325]:
health_data.iloc[:3, :2]

subject      Bob      
type          HR  Temp
year visit            
2013 1      25.0  36.9
     2      27.0  36.5
2014 1      24.0  38.1

In [326]:
health_data.columns

MultiIndex([(  'Bob',   'HR'),
            (  'Bob', 'Temp'),
            ('Guido',   'HR'),
            ('Guido', 'Temp'),
            (  'Sue',   'HR'),
            (  'Sue', 'Temp')],
           names=['subject', 'type'])

In [328]:
# Can pass a tuple of multiple indices:
health_data.loc[:, ('Bob', 'HR')]

year  visit
2013  1        25.0
      2        27.0
2014  1        24.0
      2        52.0
Name: (Bob, HR), dtype: float64

<h3>Rearranging Multi-Indices</h3>

One of the keys of working with multiply indexed data is knowing how to effectively transform the data. There are a number of operations that will preserve all the information in the dataset, but rearrange it for the purposes of various computations. We briefly saw this with the `stack()` and `unstack()` methods, but there are many more ways to finely control the rearrangement of data between hierarchical indices and columns. We'll explore some here:

<b>Sorted and unsorted indices</b>

<i>Many of the `MultiIndex` slicing operations will fail if the index is not sorted.</i>

In [336]:
# indices are not lexographically sorted
index = pd.MultiIndex.from_product([['a', 'c', 'b'], [1, 2]])
data = pd.Series(np.random.rand(6), index=index)
data.index.names = ['char', 'int']
data

char  int
a     1      0.626410
      2      0.971261
c     1      0.787566
      2      0.840063
b     1      0.880999
      2      0.187286
dtype: float64

In [338]:
try:
    data['a':'b']
except KeyError as e:
    print(type(e))
    print(e)

<class 'pandas.errors.UnsortedIndexError'>
'Key length (1) was greater than MultiIndex lexsort depth (0)'


* So, the `KeyError` was a result of the `MultiIndex` not being sorted. Note that `data['a':'c']` would also fail here due to the index not being sorted. Any partial slice will raise a KeyError.
* Partial slices and other similar operations require the levels in the `MultiIndex` to be sorted (i.e., lexographical) order. Pandas provides functions like `sort_index()` and `sortlevel()` for `DataFrame` objects. The simplest is `sort_index()`:

In [339]:
data = data.sort_index()
data

char  int
a     1      0.626410
      2      0.971261
b     1      0.880999
      2      0.187286
c     1      0.787566
      2      0.840063
dtype: float64

In [340]:
# now we can do a partial slice without KeyError
data['a':'b']

char  int
a     1      0.626410
      2      0.971261
b     1      0.880999
      2      0.187286
dtype: float64

* <b>Stacking and unstacking indices</b>

As we briefly saw before, it is possible to convert a dataset from a stacked multi-index to a simple two-dimensional representation, optionally specifying the level to use:

In [341]:
pop

state       year
California  2000    33871648
            2010    37253956
New York    2000    18976457
            2010    19378102
Texas       2000    20851820
            2010    25145561
dtype: int64

In [344]:
pop.unstack(level=0)

state,California,New York,Texas
year,,,
2000,33871648,18976457,20851820
2010,37253956,19378102,25145561


In [345]:
pop.unstack(level=1)

year,2000,2010
state,,
California,33871648,37253956
New York,18976457,19378102
Texas,20851820,25145561


* The opposite of `stack()` is `unstack()` and can be used to recover the original series:

In [346]:
pop.unstack().stack()

state       year
California  2000    33871648
            2010    37253956
New York    2000    18976457
            2010    19378102
Texas       2000    20851820
            2010    25145561
dtype: int64

* <b> Index setting and resetting (flattening / unflattening data)</b>

Another way to rearrange hierarchical data is to turn the index labels into columns; this can be accomplished with the `reset_index()` method. Calling this on the population dictionary will result in a `DataFrame` with a <i>state</i> and <i>year</i> column holding the information that was formerly in the index. For clarity, we can optionally specify the name of the data for the column representation:

In [347]:
pop

state       year
California  2000    33871648
            2010    37253956
New York    2000    18976457
            2010    19378102
Texas       2000    20851820
            2010    25145561
dtype: int64

In [348]:
type(pop)

pandas.core.series.Series

In [349]:
pop_flat = pop.reset_index(name='population')
pop_flat

,state,year,population
0,California,2000,33871648
1,California,2010,37253956
2,New York,2000,18976457
3,New York,2010,19378102
4,Texas,2000,20851820
5,Texas,2010,25145561


In [350]:
type(pop_flat)

pandas.core.frame.DataFrame

* Often when working with data in the real-world, the raw input looks like this and it's useful to build a `MultiIndex` from the column views. This can be done with the `set_index()` method of the `DataFrame`, which returns a multiply indexed `DataFrame`.
* This type of reindexing is one of the more useful patterns when encountering real-world flattened datasets

In [351]:
pop_flat

,state,year,population
0,California,2000,33871648
1,California,2010,37253956
2,New York,2000,18976457
3,New York,2010,19378102
4,Texas,2000,20851820
5,Texas,2010,25145561


In [353]:
pop_flat.set_index(['state', 'year'])

population
state      year            
California 2000    33871648
           2010    37253956
New York   2000    18976457
           2010    19378102
Texas      2000    20851820
           2010    25145561

<h3>Data Aggregations on Multi-Indices</h3>

We've gone over some Pandas built-in data aggregation methods, such as `mean()`, `sum()`, and `max()`. For hierarchically indexed data, these can be passed a `level` parameter that controls which subset of the data the aggregate is computed on:

In [355]:
health_data

subject      Bob       Guido         Sue      
type          HR  Temp    HR  Temp    HR  Temp
year visit                                    
2013 1      25.0  36.9  49.0  37.3  41.0  34.7
     2      27.0  36.5  33.0  37.6  37.0  35.2
2014 1      24.0  38.1  54.0  34.7  35.0  38.4
     2      52.0  36.1  40.0  36.1  40.0  37.4

In [358]:
# name the index level you'd like to explore, in this case the year
data_mean = health_data.mean(level='year')
data_mean

subject   Bob       Guido          Sue       
type       HR  Temp    HR   Temp    HR   Temp
year                                         
2013     26.0  36.7  41.0  37.45  39.0  34.95
2014     38.0  37.1  47.0  35.40  37.5  37.90

In [359]:
# by also using the axis keyword, we can take the mean amoung levels on the
# columns as well
data_mean.mean(axis=1, level='type')

type,HR,Temp
year,,
2013,35.333333,36.366667
2014,40.833333,36.800000


In [363]:
# without the level arg, all of the numbers are averaged, without
# regard to the 'type' (HR or Temp)
data_mean.mean(axis=1)

year
2013    35.850000
2014    38.816667
dtype: float64

* Note that this syntax is actually a shortcut to the `GroupBy` functionality, which we will discuss shortly. Many real-world datasets have similar hierarchical structure. 

<h3>Panel Data</h3>

* Pandas has a few other fundamental data structures that we have not discussed yet, namely the `pd.Panel` and `pd.Panel4D` objects. These can be thought of as three-dimensional and four-dimensional generalizations of the (one-dimensional) `Series` and (two-dimensional) `DataFrame` structures.
* Once familiar with the indexing and manipulation of data in a `Series` and `DataFrame`, `Panel` and `Panel4D` are relatively straightforward to use. The `loc` and `iloc` indexers extend readily to these higher-dimensional structures.
* We won't cover the panel structures further here, since multi-indexing is often more useful and conceptually simpler representation for higher-dimensional data.
* <b>Panel data is fundamentally a dense data representation (has mostly non-zero values), while multi-indexing is fundamentally a sparse data representation (has mostly zero values). As the number of dimensions increases, the dense representation can become very inefficient for the majority of real-world datasets.</b>
* For occasional specialized applications, however, these structures can be useful.

<h3>Combining Datasets: Concat and Append</h3>

* Some of the most interesting data come from combining different data sources. These operations can involve anything from very straightforward concatenations of the two different datasets, to more complicated database-style joins and merges that handle overlaps between datasets.
* `Series` and `DataFrames` are built with this type of operation in mind, and Pandas includes functions and methods that make this sort of data wrangling fast and straightforward
* Here we'll look at simple concatenation of `Series` and `DataFrames` with the `pd.concat()` function

In [367]:
def make_df(cols, ind):
    """Quickly make a DataFrame"""
    data = {c: [str(c) + str(i) for i in ind] for c in cols}
    return pd.DataFrame(data, ind)

In [368]:
make_df('ABC', range(3))

,A,B,C
0,A0,B0,C0
1,A1,B1,C1
2,A2,B2,C2


* Concatenation of `Series` and `DataFrame` objects is very similar to concatenation of NumPy arrays, which can be done via the `np.concatenate()` function

In [370]:
x = [1, 2, 3]
y = [4, 5, 6]
z = [7, 8, 9]
np.concatenate([x, y, z])

array([1, 2, 3, 4, 5, 6, 7, 8, 9])

In [371]:
# you can also specify the axis along which the result will be concatenated
x = [[1, 2],
     [3, 4]]
np.concatenate([x, x])

array([[1, 2],
       [3, 4],
       [1, 2],
       [3, 4]])

In [372]:
np.concatenate([x, x], axis=1)

array([[1, 2, 1, 2],
       [3, 4, 3, 4]])

In [373]:
ser1 = pd.Series(['A', 'B', 'C'], index=[1, 2, 3])
ser2 = pd.Series(['D', 'E', 'F'], index=[4, 5, 6])
pd.concat([ser1, ser2])

1    A
2    B
3    C
4    D
5    E
6    F
dtype: object

In [387]:
# can also concatenate higher-dimensional objects, such as DataFrames
df1 = make_df('AB', [1, 2])
df2 = make_df('AB', [3, 4])
print('df1\n\n', df1); print('\n\ndf2\n\n', df2); print('\n\npd.concat\n\n', pd.concat([df1, df2]))

df1

     A   B
1  A1  B1
2  A2  B2


df2

     A   B
3  A3  B3
4  A4  B4


pd.concat

     A   B
1  A1  B1
2  A2  B2
3  A3  B3
4  A4  B4


* By default, the concatenation takes place row-wise within the `DataFrame` (i.e., axis=0). Just like `np.concatenate()`, `pd.concat()` allows specification of an axis along which concatenation will take place. 

In [394]:
df3 = make_df('AB', [0, 1])
df4 = make_df('CD', [0, 1])
# axis='columns' is equivalent to axis=1; axis=0 refers to 'rows'
print(df3); print(df4); print(pd.concat([df3, df4], axis='columns'))

    A   B
0  A0  B0
1  A1  B1
    C   D
0  C0  D0
1  C1  D1
    A   B   C   D
0  A0  B0  C0  D0
1  A1  B1  C1  D1


* `pd.concat?` function signature shows default axis=0 (rows)
```python
pd.concat?

Signature:
pd.concat(
    objs: Union[Iterable[ForwardRef('NDFrame')], Mapping[Union[Hashable, NoneType], ForwardRef('NDFrame')]],
    axis=0,
    join='outer',
    ignore_index: bool = False,
    keys=None,
    levels=None,
    names=None,
    verify_integrity: bool = False,
    sort: bool = False,
    copy: bool = True,
) -> Union[ForwardRef('DataFrame'), ForwardRef('Series')]
```

* <b>What about duplicate indices?</b>

* An important difference between `np.concatenate()` and `pd.concat()` is that Pandas concatenation <i>preserves indices</i>, even if the result will have duplicate indices.

In [395]:
x = make_df('AB', [0, 1])
y = make_df('AB', [2, 3])
# duplicate the indices
y.index = x.index

print(x); print(y); print(pd.concat([x, y]))

    A   B
0  A0  B0
1  A1  B1
    A   B
0  A2  B2
1  A3  B3
    A   B
0  A0  B0
1  A1  B1
0  A2  B2
1  A3  B3


* Notice the repeated indices in the result. While this is valid with `DataFrame` objects, it is often undesirable. `pd.concat()` gives us a few ways to handle it.

* <b>Catching repeats as an error:</b> If you'd like to simply verify that the indices in the result of `pd.concat()` do not overlap, you can specify the `verify_integrity` flag. If set to `True`, the concatenation will raise an exception if there are duplicate indices. Example below where we'll catch and print the error message:

In [399]:
try:
    pd.concat([x, y], verify_integrity=True)
except ValueError as e:
    print('ValueError:', e)

ValueError: Indexes have overlapping values: Int64Index([0, 1], dtype='int64')


* <b>Ignoring the index</b>: Sometimes the index itself does not matter, and you would prefer it to simply be ignored. Specify this using the `ignore_index` flag. Concatenation will create a new integer index for the resulting `Series`:

In [400]:
print(x); print(y); print(pd.concat([x, y], ignore_index=True))

    A   B
0  A0  B0
1  A1  B1
    A   B
0  A2  B2
1  A3  B3
    A   B
0  A0  B0
1  A1  B1
2  A2  B2
3  A3  B3


* <b>Adding MultiIndex keys</b>: Another alternative is to use the `keys` option to specify a label for the data sources; the result will be an index that is a hierarchically indexed series containing the data:

In [405]:
# The result of concatenation is a multiply indexed DataFrame
print(x); print(y); print(pd.concat([x, y], keys=['x_df', 'y_df']))

    A   B
0  A0  B0
1  A1  B1
    A   B
0  A2  B2
1  A3  B3
         A   B
x_df 0  A0  B0
     1  A1  B1
y_df 0  A2  B2
     1  A3  B3


In [406]:
conc = pd.concat([x, y], keys=['x_df', 'y_df'])
type(conc)

pandas.core.frame.DataFrame

In [407]:
conc.index

MultiIndex([('x_df', 0),
            ('x_df', 1),
            ('y_df', 0),
            ('y_df', 1)],
           )

* <b>Concatenation with joins</b>

In [409]:
df5 = make_df('ABC', [1, 2])
df6 = make_df('BCD', [3, 4])
print(df5); print(df6); print(pd.concat([df5, df6]))

    A   B   C
1  A1  B1  C1
2  A2  B2  C2
    B   C   D
3  B3  C3  D3
4  B4  C4  D4
     A   B   C    D
1   A1  B1  C1  NaN
2   A2  B2  C2  NaN
3  NaN  B3  C3   D3
4  NaN  B4  C4   D4


* By default, the entries for which no data is available are filled with NA values. To change this, we can specify one of several options for the `join` and `join_axes` parameters of the concatenate function. By default, the join is a union of the input columns `(join='outer')`, but we can change this to an intersection of the columns using `join='inner'`:

In [410]:
print(df5); print(df6);
print(pd.concat([df5, df6], join='inner'))

    A   B   C
1  A1  B1  C1
2  A2  B2  C2
    B   C   D
3  B3  C3  D3
4  B4  C4  D4
    B   C
1  B1  C1
2  B2  C2
3  B3  C3
4  B4  C4


* <b>The `append()` method</b>

Because direct array concatenation is so common, `Series` and `DataFrame` objects have an `append()` method that can accomplish the same thing in fewer keystrokes. Rather than calling `pd.concat([df1, df2])`, you can simply call `df1.append(df2)`.

In [419]:
print(df1); print(df2)

    A   B
1  A1  B1
2  A2  B2
    A   B
3  A3  B3
4  A4  B4


In [420]:
df1.append(df2)

,A,B
1,A1,B1
2,A2,B2
3,A3,B3
4,A4,B4


* Note that unlike the `append()` and `extend()` methods of Python lists, the `append()` method in Pandas does not modify the original object -- it creates a new object with the combined data. It is also not very efficient, because it creates a new index <i>and</i> data buffer.
* <b>Therefore, if you plan to do multiple `append` operations, it is generally better to build a list of `DataFrames` and pass them all at once to the `concat()` function.</b>

<h3>Combining Datasets: Merge and Join</h3>

* An essential feature offered by Pandas is its high-performance, in-memory join and merge operations. The main interface for this is the `pd.merge()` function.
* The behavior implemented in `pd.merge()` is a subset of what is known as relational algebra, which is a formal set of rules for manipulating relational data, and forms the conceptual foundation of operations available in most databases.
* The strength of the relational algebra approach is that it proposes several primitive operations, which become the building blocks of more complicated operations on any dataset. A wide range of fairly complicated composite operations can be performed with these fundamental operations implemented efficiently.


<h3>Categories of Joins</h3>

* The `pd.merge()` function implements a number of types of joins: the <i>one-to-one, many-to-one, and many-to-many</i> joins. All three types of joins are accessed via an identical call to the `pd.merge()` interface; we'll see examples of all three types of merges below.

<b>One-to-one joins</b>

* Similar to column-wise concatenation from earlier

In [421]:
df1 = pd.DataFrame({'employee': ['Bob', 'Jake', 'Lisa', 'Sue'],
                    'group': ['Accounting', 'Engineering', 'Engineering', 'HR']})
df2 = pd.DataFrame({'employee': ['Lisa', 'Bob', 'Jake', 'Sue'],
                    'hire_date': [2004, 2008, 2012, 2014]})

In [422]:
print(df1); print(df2)

  employee        group
0      Bob   Accounting
1     Jake  Engineering
2     Lisa  Engineering
3      Sue           HR
  employee  hire_date
0     Lisa       2004
1      Bob       2008
2     Jake       2012
3      Sue       2014


In [423]:
# combine the information into a single df
df3 = pd.merge(df1, df2)
df3

,employee,group,hire_date
0,Bob,Accounting,2008
1,Jake,Engineering,2012
2,Lisa,Engineering,2004
3,Sue,HR,2014


* How did this join happen?
* The `pd.merge()` function recognized that each `DataFrame` has an "employee" column, and automatically joins using this column as a key.
* Note that the order of entries in each column does not matter: in this case, the order of the "employee" column differs between `df1` and `df2`, and the `pd.merge()` function correctly accounts for this.
* Also note that merge in general discards the index, except in the special case of merges by index.

<b>Many-to-one joins</b>

* These are joins in which one of the two key columns contains duplicate entries. 

In [424]:
df4 = pd.DataFrame({'group': ['Accounting', 'Engineering', 'HR'],
                    'supervisor': ['Carly', 'Guido', 'Steve']})
print(df3); print(df4)

  employee        group  hire_date
0      Bob   Accounting       2008
1     Jake  Engineering       2012
2     Lisa  Engineering       2004
3      Sue           HR       2014
         group supervisor
0   Accounting      Carly
1  Engineering      Guido
2           HR      Steve


In [425]:
print(pd.merge(df3, df4))

  employee        group  hire_date supervisor
0      Bob   Accounting       2008      Carly
1     Jake  Engineering       2012      Guido
2     Lisa  Engineering       2004      Guido
3      Sue           HR       2014      Steve


* The resulting `DataFrame` has an additional column with the "supervisor" information, where the information is repeated in one or more locations as required by the inputs

<b>Many-to-many joins</b>

* If the key column in both the left and right array contains duplicates, then the result is a many-to-many merge.

In [437]:
df5 = pd.DataFrame({'group': ['Accounting', 'Accounting',
                              'Engineering', 'Engineering', 'HR', 'HR'],
                    'skills': ['math', 'spreadsheets', 'coding', 'linux',
                                      'spreadsheets', 'organization']})

In [438]:
print(df1); print(df5)

  employee        group
0      Bob   Accounting
1     Jake  Engineering
2     Lisa  Engineering
3      Sue           HR
         group        skills
0   Accounting          math
1   Accounting  spreadsheets
2  Engineering        coding
3  Engineering         linux
4           HR  spreadsheets
5           HR  organization


In [439]:
pd.merge(df1, df5)

,employee,group,skills
0,Bob,Accounting,math
1,Bob,Accounting,spreadsheets
2,Jake,Engineering,coding
3,Jake,Engineering,linux
4,Lisa,Engineering,coding
5,Lisa,Engineering,linux
6,Sue,HR,spreadsheets
7,Sue,HR,organization


<h3>Specification of the Merge Key</h3>

* Datasets are rarely as clean as the one's we're working with here.
* The default behavior of `pd.merge()` is what we've already seen here: it looks for one or more matching column names between the two inputs, and uses this as the key. However, often the column names will not match so nicely, so there are other options.

In [440]:
print(df1); print(df2)

  employee        group
0      Bob   Accounting
1     Jake  Engineering
2     Lisa  Engineering
3      Sue           HR
  employee  hire_date
0     Lisa       2004
1      Bob       2008
2     Jake       2012
3      Sue       2014


In [443]:
# both DataFrames have to have the specified column name
pd.merge(df1, df2, on='employee')

,employee,group,hire_date
0,Bob,Accounting,2008
1,Jake,Engineering,2012
2,Lisa,Engineering,2004
3,Sue,HR,2014


* <b>The left_on and right_on keywords</b>:
* If the column names are different, use these.
* The result will have a redundant column, which we can drop if desired.

In [444]:
df3 = pd.DataFrame({'name': ['Bob', 'Jake', 'Lisa', 'Sue'],
                    'salary': [70000, 80000, 120000, 90000]})

In [445]:
print(df1); print(df3)

  employee        group
0      Bob   Accounting
1     Jake  Engineering
2     Lisa  Engineering
3      Sue           HR
   name  salary
0   Bob   70000
1  Jake   80000
2  Lisa  120000
3   Sue   90000


In [446]:
pd.merge(df1, df3, left_on='employee', right_on='name')

,employee,group,name,salary
0,Bob,Accounting,Bob,70000
1,Jake,Engineering,Jake,80000
2,Lisa,Engineering,Lisa,120000
3,Sue,HR,Sue,90000


In [448]:
# Drop the redundant column
pd.merge(df1, df3, left_on='employee', right_on='name').drop('name', axis=1)

,employee,group,salary
0,Bob,Accounting,70000
1,Jake,Engineering,80000
2,Lisa,Engineering,120000
3,Sue,HR,90000


* `left_index` and `right_index` keywords
* Rather than merging on a column, you can merge on an index:

In [449]:
df1a = df1.set_index('employee')
df2a = df2.set_index('employee')
print(df1a); print(df2a)

                group
employee             
Bob        Accounting
Jake      Engineering
Lisa      Engineering
Sue                HR
          hire_date
employee           
Lisa           2004
Bob            2008
Jake           2012
Sue            2014


In [450]:
pd.merge(df1a, df2a, left_index=True, right_index=True)

,group,hire_date
employee,,
Bob,Accounting,2008
Jake,Engineering,2012
Lisa,Engineering,2004
Sue,HR,2014


* `DataFrames` implement the `join()` method, which performs a merge that defaults to joining on indices:

In [451]:
df1a.join(df2a)

,group,hire_date
employee,,
Bob,Accounting,2008
Jake,Engineering,2012
Lisa,Engineering,2004
Sue,HR,2014


* You can even mix indices and columns by combining `left_index` with `right_on` or `left_on` with `right_index`:

In [452]:
print(df1a); print(df3)

                group
employee             
Bob        Accounting
Jake      Engineering
Lisa      Engineering
Sue                HR
   name  salary
0   Bob   70000
1  Jake   80000
2  Lisa  120000
3   Sue   90000


In [456]:
pd.merge(df1a, df3, left_index=True, right_on='name')

,group,name,salary
0,Accounting,Bob,70000
1,Engineering,Jake,80000
2,Engineering,Lisa,120000
3,HR,Sue,90000


In [455]:
pd.merge(df1a, df3, left_on='employee', right_on='name')

,group,name,salary
0,Accounting,Bob,70000
1,Engineering,Jake,80000
2,Engineering,Lisa,120000
3,HR,Sue,90000


<h3>Specifying Set Arithmetic for Joins</h3>

* In all previous examples, we glossed over an important consideration: the type of join. This comes up when a value appears in one key column but not the other.

In [459]:
df6 = pd.DataFrame({'name': ['Peter', 'Paul', 'Mary'],
                    'food': ['fish', 'beans', 'bread']},
                    columns=['name', 'food'])
df7 = pd.DataFrame({'name': ['Mary', 'Joseph'],
                    'drink': ['wine', 'beer']},
                    columns=['name', 'drink'])

In [460]:
print(df6); print(df7)

    name   food
0  Peter   fish
1   Paul  beans
2   Mary  bread
     name drink
0    Mary  wine
1  Joseph  beer


In [461]:
pd.merge(df6, df7)

,name,food,drink
0,Mary,bread,wine


* By default, the result contains only the <i>intersection</i> of the two sets of inputs; this is what is known as an `inner join`. We can specify this by using the `how` keyword.
* Other options include `outer`, `left`, and `right`.
* An `outer` join returns a join over the union of the input columns, and fills in all missing values with NAs.

In [462]:
pd.merge(df6, df7, how='inner')

,name,food,drink
0,Mary,bread,wine


In [463]:
pd.merge(df6, df7, how='outer')

,name,food,drink
0,Peter,fish,NaN
1,Paul,beans,NaN
2,Mary,bread,wine
3,Joseph,NaN,beer


* The `left` and `right` joins return a join over the left entries and right entries, respectively.

In [464]:
pd.merge(df6, df7, how='left')

,name,food,drink
0,Peter,fish,NaN
1,Paul,beans,NaN
2,Mary,bread,wine


* If there are overlapping column names with the `merge()` function, `DataFrames` will append a suffix `_x` or `_y` to make the output columns unique:

In [465]:
df8 = pd.DataFrame({'name': ['Bob', 'Jake', 'Lisa', 'Sue'],
                    'rank': [1, 2, 3, 4]})
df9 = pd.DataFrame({'name': ['Bob', 'Jake', 'Lisa', 'Sue'],
                    'rank': [3, 1, 4, 2]})

In [466]:
pd.merge(df8, df9, on='name')

,name,rank_x,rank_y
0,Bob,1,3
1,Jake,2,1
2,Lisa,3,4
3,Sue,4,2


In [467]:
# you can specify the suffix
pd.merge(df8, df9, on='name', suffixes=['_L', '_R'])

,name,rank_L,rank_R
0,Bob,1,3
1,Jake,2,1
2,Lisa,3,4
3,Sue,4,2


<h3>US States Data Example DataFrames</h3>

In [468]:
# shell commands to download US States Data
!curl -O https://raw.githubusercontent.com/jakevdp/data-USstates/master/state-population.csv
!curl -O https://raw.githubusercontent.com/jakevdp/data-USstates/master/state-areas.csv
!curl -O https://raw.githubusercontent.com/jakevdp/data-USstates/master/state-abbrevs.csv

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 57935  100 57935    0     0  60785      0 --:--:-- --:--:-- --:--:-- 60792
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100   835  100   835    0     0   1379      0 --:--:-- --:--:-- --:--:--  1377
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100   872  100   872    0     0   1579      0 --:--:-- --:--:-- --:--:--  1579


In [469]:
pop = pd.read_csv('state-population.csv')
areas = pd.read_csv('state-areas.csv')
abbrevs = pd.read_csv('state-abbrevs.csv')

In [471]:
print(pop.head()); print(areas.head()); print(abbrevs.head())

  state/region     ages  year  population
0           AL  under18  2012   1117489.0
1           AL    total  2012   4817528.0
2           AL  under18  2010   1130966.0
3           AL    total  2010   4785570.0
4           AL  under18  2011   1125763.0
        state  area (sq. mi)
0     Alabama          52423
1      Alaska         656425
2     Arizona         114006
3    Arkansas          53182
4  California         163707
        state abbreviation
0     Alabama           AL
1      Alaska           AK
2     Arizona           AZ
3    Arkansas           AR
4  California           CA


In [477]:
# Rank US states by their 2010 population density

merged = pd.merge(pop, abbrevs, left_on='state/region', right_on='abbreviation')
merged = merged.drop('abbreviation', axis=1) # drop duplicate column
merged.head()

,state/region,ages,year,population,state
0,AL,under18,2012,1117489.0,Alabama
1,AL,total,2012,4817528.0,Alabama
2,AL,under18,2010,1130966.0,Alabama
3,AL,total,2010,4785570.0,Alabama
4,AL,under18,2011,1125763.0,Alabama


In [478]:
# Let's check if there were any mismatches by checking rows with nulls
merged.isnull().any()

state/region    False
ages            False
year            False
population      False
state           False
dtype: bool

In [480]:
final = pd.merge(merged, areas, on='state', how='left')
final.head()

,state/region,ages,year,population,state,area (sq. mi)
0,AL,under18,2012,1117489.0,Alabama,52423
1,AL,total,2012,4817528.0,Alabama,52423
2,AL,under18,2010,1130966.0,Alabama,52423
3,AL,total,2010,4785570.0,Alabama,52423
4,AL,under18,2011,1125763.0,Alabama,52423


In [481]:
# check again for mismatches
final.isnull().any()

state/region     False
ages             False
year             False
population       False
state            False
area (sq. mi)    False
dtype: bool

* High performance `query` function in pandas:

In [483]:
data2010 = final.query("year == 2010 & ages == 'total'")
data2010.head()

,state/region,ages,year,population,state,area (sq. mi)
3,AL,total,2010,4785570.0,Alabama,52423
91,AK,total,2010,713868.0,Alaska,656425
101,AZ,total,2010,6408790.0,Arizona,114006
189,AR,total,2010,2922280.0,Arkansas,53182
197,CA,total,2010,37333601.0,California,163707


In [484]:
# compute population density and display it in order
data2010.set_index('state', inplace=True)
density = data2010['population'] / data2010['area (sq. mi)']

In [485]:
density.sort_values(ascending=False, inplace=True)
density.head()

state
District of Columbia    8898.897059
New Jersey              1009.253268
Rhode Island             681.339159
Connecticut              645.600649
Massachusetts            621.815538
dtype: float64

In [486]:
density.tail()

state
South Dakota    10.583512
North Dakota     9.537565
Montana          6.736171
Wyoming          5.768079
Alaska           1.087509
dtype: float64

* This type of messy data merging is common when working with real-world data sources.

<h3>Aggregation and Grouping</h3>

* An essential piece of analysis of large data is efficient summarization: computing aggregations like `sum()`, `mean()`, `median()`, `min()`, and `max()`, in which a single number gives insight into the nature of a potentially large dataset.

<h3>Seaborn Planets Data</h3>

In [488]:
import seaborn as sns

planets = sns.load_dataset('planets')
planets.shape

(1035, 6)

In [489]:
planets.head()

,method,number,orbital_period,mass,distance,year
0,Radial Velocity,1,269.300,7.10,77.40,2006
1,Radial Velocity,1,874.774,2.21,56.95,2008
2,Radial Velocity,1,763.000,2.60,19.84,2011
3,Radial Velocity,1,326.030,19.40,110.62,2007
4,Radial Velocity,1,516.220,10.50,119.47,2009


<h3>Simple Aggregation in Pandas</h3>

* For a Pandas Series, aggregates return a single value, just like with a one-dimensional NumPy array:

In [490]:
rng = np.random.RandomState(42)
ser = pd.Series(rng.rand(5))
ser

0    0.374540
1    0.950714
2    0.731994
3    0.598658
4    0.156019
dtype: float64

In [491]:
ser.sum()

2.811925491708157

In [492]:
ser.mean()

0.5623850983416314

In [493]:
df = pd.DataFrame({'A': rng.rand(5),
                   'B': rng.rand(5)})
df

,A,B
0,0.155995,0.020584
1,0.058084,0.969910
2,0.866176,0.832443
3,0.601115,0.212339
4,0.708073,0.181825


* By default, a `DataFrame` aggregate will return results within each column:

In [494]:
df.mean()

A    0.477888
B    0.443420
dtype: float64

* By specifying the `axis` argument, you can instead aggregate within each row:

In [495]:
df.mean(axis='columns')

0    0.088290
1    0.513997
2    0.849309
3    0.406727
4    0.444949
dtype: float64

* Pandas has a `describe()` convenience method that computes several common aggregates for each column and returns the result. 

In [497]:
# drop rows with missing values for now
planets.dropna().describe()

,number,orbital_period,mass,distance,year
count,498.00000,498.000000,498.000000,498.000000,498.000000
mean,1.73494,835.778671,2.509320,52.068213,2007.377510
std,1.17572,1469.128259,3.636274,46.596041,4.167284
min,1.00000,1.328300,0.003600,1.350000,1989.000000
25%,1.00000,38.272250,0.212500,24.497500,2005.000000
50%,1.00000,357.000000,1.245000,39.940000,2009.000000
75%,2.00000,999.600000,2.867500,59.332500,2011.000000
max,6.00000,17337.500000,25.000000,354.000000,2014.000000


* From this, we can see things in the `year` column like an exoplanet was discovered as far back as 1989, but half of all known exoplanets were not discovered until 2010 or later (largely due to the Kepler telescope mission).

<h3>GroupBy: Split, Apply, Combine</h3>

* Simple aggregations are helpful, but we often need to aggregate conditionally on some label or index: this is implemented into the `groupby` operation. The name comes from the SQL command, but it can help to think of it as <i>split, apply, combine.</i>
* Split: breaking up and grouping a `DataFrame` depending on the value of the specified key
* Apply: involves computing some function, usually an aggregate, transformation, or filtering, within the individual groups
* Combine: merge the results of these operations into an output array

* The power of `GroupBy` is that it abstracts these steps -- we do not need to do some kind of masking, aggregation, and merging commands. Just think about the operation as a whole.

In [499]:
df = pd.DataFrame({'key': ['A', 'B', 'C', 'A', 'B', 'C'],
                   'data': range(6)}, columns=['key', 'data'])

In [500]:
df

,key,data
0,A,0
1,B,1
2,C,2
3,A,3
4,B,4
5,C,5


In [501]:
df.groupby('key')

* Note how this returns a `DataFrameGroupBy` object rather than a set of `DataFrames`. This is where the magic is: think of it as a special view of the `DataFrame`, which can dig into the groups but does no actual computation until the aggregation is applied.

In [502]:
df.groupby('key').sum()

,data
key,
A,3
B,5
C,7


* Some of the most important operations made available by `GroupBy` are <i>aggregate, filter, transform, apply.</i> We'll look at these more later.

<b>Column Indexing</b>
* The `GroupBy` object supports column indexing in the same way as the `DataFrame`, and returns a modified `GroupBy` object:

In [503]:
planets.groupby('method')

In [504]:
# no computation is done until some aggregate is called on the object
planets.groupby('method')['orbital_period']

In [505]:
planets.groupby('method')['orbital_period'].median()

method
Astrometry                         631.180000
Eclipse Timing Variations         4343.500000
Imaging                          27500.000000
Microlensing                      3300.000000
Orbital Brightness Modulation        0.342887
Pulsar Timing                       66.541900
Pulsation Timing Variations       1170.000000
Radial Velocity                    360.200000
Transit                              5.714932
Transit Timing Variations           57.011000
Name: orbital_period, dtype: float64

In [ ]:
# p. 164

<h3>Review chained indexing in Pandas as well </h3>